In [1]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()

token = user_secrets.get_secret("vivqa")

username = "nmluan85"
repo_name = "ViVQA-LanguageBias"

repo_url = f"https://{token}@github.com/{username}/{repo_name}.git"
!git clone {repo_url}

Cloning into 'ViVQA-LanguageBias'...
remote: Enumerating objects: 84, done.
remote: Counting objects: 100% (84/84), done.
remote: Compressing objects: 100% (61/61), done.
remote: Total 84 (delta 30), reused 74 (delta 20), pack-reused 0 (from 0)
Receiving objects: 100% (84/84), 85.16 KiB | 2.37 MiB/s, done.
Resolving deltas: 100% (30/30), done.


In [2]:
!pip install -r /kaggle/working/ViVQA-LanguageBias/requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 266.3/266.3 kB 8.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 190.3/190.3 kB 14.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 83.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 549.1/549.1 kB 37.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.2/71.2 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 70.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10

In [3]:
!pip install torchscale
!pip install mlflow efficientnet_pytorch underthesea
!pip install timm==1.0.13

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.3/53.3 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 36.8 MB/s eta 0:00:00
  Attempting uninstall: timm
    Found existing installation: timm 0.6.13
    Uninstalling timm-0.6.13:
      Successfully uninstalled timm-0.6.13
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchscale 0.3.0 requires timm==0.6.13, but you have timm 1.0.13 which is incompatible.


In [4]:
import sys
import os

repo_path = "/kaggle/working/ViVQA-LanguageBias"
if repo_path not in sys.path:
    sys.path.append(repo_path)

os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["CUDA_VISIBLE_DEVICES"] = "0" # Use only the first GPU
os.environ['MLFLOW_EXPERIMENT_NAME'] = 'mlflow-vivqa'

In [5]:
import argparse
base_path = "/kaggle/working/ViVQA-LanguageBias"
opt = argparse.Namespace()

# Training Parameters
opt.output_dir = "./output"
opt.log_level = "passive"
opt.lr_scheduler_type = "cosine"
opt.warmup_ratio = 0.1
opt.logging_strategy = "epoch"
opt.save_strategy = "epoch"
opt.save_total_limit = 1
opt.train_batch_size = 16
opt.eval_batch_size = 16
opt.epochs = 2
opt.learning_rate = 3e-5
opt.weight_decay = 0.01
opt.workers = 2

# Dataset Paths (Absolute Kaggle Paths)
opt.image_path = f"/kaggle/input/datasets/luan85/viet-vqa/images"
opt.ans_path = f"/kaggle/input/datasets/luan85/viet-vqa/vocab.json"
opt.train_path = f"/kaggle/input/datasets/luan85/viet-vqa/train.csv"
opt.val_path = f"{base_path}/data/ViVQA-csv/val.csv"
opt.test_path = f"/kaggle/input/datasets/luan85/viet-vqa/test.csv"
opt.feature_paths = f"/kaggle/working/"

# Model Architecture
opt.drop_path_rate = 0.3
opt.encoder_layers = 6
opt.encoder_attention_heads_layers = 6
opt.classes = 353

In [6]:
import torch
from torch.utils.data import DataLoader
from PIL import Image
import os
import glob
from tqdm import tqdm
import numpy as np
import h5py


def extract_features(vs_encoder, opt, model_name):
    file_paths = sorted(glob.glob(os.path.join(opt.image_path, '*.jpg')))
    features_shape = (len(file_paths), 32, 768)

    with h5py.File(f'{opt.feature_paths}/{model_name}.hdf5', 'w') as f:
        features = f.create_dataset('features', shape=features_shape, dtype='float16')
        img_ids = f.create_dataset('ids', shape=(len(file_paths),), dtype='int32')

        j = k = 0
        data_loader = DataLoader(file_paths, batch_size=32, shuffle=False, num_workers=4)
        for paths in tqdm(data_loader):
            ids = [os.path.basename(path).split('.')[0] for path in paths]
            imgs = [Image.open(path) for path in paths]
            
            with torch.no_grad():
                out = vs_encoder(*imgs)
                
            k = j + len(paths)
            features[j:k] = out.cpu().numpy().astype('float16')
            img_ids[j:k] = np.array(ids, dtype='int32')
            j = k

In [7]:
from VisionEmbedding import EfficientnetExtractor, Blip2ViTExtractor
# model_efficientnet = EfficientnetExtractor("efficientnet-b7")
model_blip2 = Blip2ViTExtractor()
# extract_features(model_efficientnet, opt, "efficientnet-b7")
extract_features(model_blip2, opt, "blip-2")

2026-03-07 17:11:05.127742: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772903465.334959      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772903465.391356      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772903465.868640      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772903465.868677      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772903465.868680      24 computation_placer.cc:177] computation placer alr

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

Downloading: https://storage.googleapis.com/sfr-vision-language-research/LAVIS/models/BLIP2/eva_vit_g.pth to /root/.cache/torch/hub/checkpoints/eva_vit_g.pth


... (2 outputs left out)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertLMHeadModel were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['bert.encoder.layer.0.crossattention.output.LayerNorm.bias', 'bert.encoder.layer.0.crossattention.output.LayerNorm.weight', 'bert.encoder.layer.0.crossattention.output.dense.bias', 'bert.encoder.layer.0.crossattention.output.dense.weight', 'bert.encoder.layer.0.crossattention.self.key.bias', 'bert.encoder.layer.0.crossattention.self.key.weight', 'bert.encoder.layer.0.crossattention.self.query.bias', 'bert.encoder.layer.0.crossattention.self.query.weight', 'bert.encoder.layer.0.crossattention.self.value.bias', 'bert.encoder.layer.0.crossattention.self.value.weight', 'bert.encoder.layer.0.intermediate_query.dense.bias', 'bert.encoder.layer.0.intermediate_query.dense.weight', 'bert.encoder.layer.0.output_query.LayerNorm.bias', 'bert.encoder.layer.0.output_query.LayerNorm.weight', 'bert.encoder.layer.0.output_query.dense.bias', 'bert.encoder.layer.0.output_query.d

Downloading: https://storage.googleapis.com/sfr-vision-language-research/LAVIS/models/BLIP2/blip2_finetune_coco.pth to /root/.cache/torch/hub/checkpoints/blip2_finetune_coco.pth


100%|██████████| 4.37G/4.37G [00:24<00:00, 193MB/s]
  0%|          | 0/323 [00:00<?, ?it/s]/kaggle/working/ViVQA-LanguageBias/lavis/models/blip2_model/blip2.py:43: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  return torch.cuda.amp.autocast(dtype=dtype)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:85: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(
100%|████